# Employee Attrition Analytics
### Kayfa AI & Data Analytics Internship · Week 1 · Data Analytics Track
---
*End-to-end analytics pipeline: load, clean, explore, and surface HR insights from 74,498 synthetic employee records.*


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")

# Global visual config
COLORS   = {"Left": "#E45756", "Stayed": "#4E79A7"}
TEMPLATE = "plotly_white"


## 1 · Load & Combine
The dataset ships as two splits — `train.csv` and `test.csv`.  
We merge them into a single DataFrame so every analysis step works on the full population.


In [ ]:
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

df_raw = pd.concat([train, test], ignore_index=True)

print(f"train   : {len(train):>6,} rows")
print(f"test    : {len(test):>6,} rows")
print(f"combined: {len(df_raw):>6,} rows  x  {df_raw.shape[1]} columns")
df_raw.sample(3, random_state=42)


## 2 · Preprocessing & Cleaning
Clean data is the foundation of honest analysis.  
We tackle four things: column names, missing values, duplicates, and data types.


In [ ]:
df = df_raw.copy()

# Normalize to snake_case: "Monthly Income" -> "monthly_income"
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print("Columns after normalization:")
print(df.columns.tolist())


In [ ]:
missing = df.isnull().sum()
print("Missing Values")
print("=" * 40)
print(missing[missing > 0].to_string() if missing.any() else "  None — dataset is complete.")

dupes = df.duplicated().sum()
print(f"\nDuplicate Rows: {dupes}")
if dupes:
    df = df.drop_duplicates().reset_index(drop=True)
    print("  Duplicates dropped.")
else:
    print("  None — nothing to drop.")


In [ ]:
# Binary flag: 1 = Left, 0 = Stayed  (used in every rate calculation)
df["attrition_flag"] = (df["attrition"] == "Left").astype(int)

# Ordinal category order — tells pandas (and plotly) how to sort these
ORDINALS = {
    "work_life_balance"   : ["Poor", "Fair", "Good", "Excellent"],
    "job_satisfaction"    : ["Low", "Medium", "High", "Very High"],
    "performance_rating"  : ["Low", "Below Average", "Average", "High"],
    "education_level"     : ["High School", "Associate Degree",
                             "Bachelor’s Degree", "Master’s Degree", "PhD"],
    "job_level"           : ["Entry", "Mid", "Senior"],
    "company_size"        : ["Small", "Medium", "Large"],
    "company_reputation"  : ["Poor", "Fair", "Good", "Excellent"],
    "employee_recognition": ["Low", "Medium", "High", "Very High"],
}

for col, order in ORDINALS.items():
    df[col] = pd.Categorical(df[col], categories=order, ordered=True)

print("Ordinal columns correctly ordered.")
print(df.dtypes)


In [ ]:
total = len(df)
left  = df["attrition_flag"].sum()
rate  = left / total * 100

print(f"{'Shape':<24} {df.shape}")
print(f"{'Total employees':<24} {total:,}")
print(f"{'Employees who left':<24} {left:,}  ({rate:.1f}%)")
print(f"{'Employees who stayed':<24} {total - left:,}  ({100-rate:.1f}%)")
df.describe().T


## 3 · Exploratory Data Analysis

**The central question:** Who leaves — and what do they have in common?

We measure attrition *rates* (%) rather than raw counts, which makes comparisons  
across groups of unequal sizes honest and meaningful.


In [ ]:
def group_attrition(col):
    """
    Compute attrition rate (%) per category of a column.
    Returns a DataFrame: [col, total, left, rate]
    """
    g = (
        df.groupby(col, observed=True)["attrition_flag"]
        .agg(total="count", left="sum")
        .reset_index()
    )
    g["rate"] = (g["left"] / g["total"] * 100).round(1)
    return g


### 3.1 · Attrition Overview — The Big Picture

In [ ]:
counts = df["attrition"].value_counts().reset_index()
counts.columns = ["status", "employees"]
overall_rate = df["attrition_flag"].mean() * 100

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "bar"}, {"type": "pie"}]],
    subplot_titles=[
        "Headcount by Status",
        f"Overall Attrition Rate: {overall_rate:.1f}%"
    ]
)

fig.add_trace(go.Bar(
    x=counts["status"],
    y=counts["employees"],
    marker_color=[COLORS[s] for s in counts["status"]],
    text=counts["employees"].apply(lambda x: f"{x:,}"),
    textposition="outside",
    showlegend=False
), row=1, col=1)

fig.add_trace(go.Pie(
    labels=counts["status"],
    values=counts["employees"],
    marker_colors=[COLORS[s] for s in counts["status"]],
    hole=0.42,
    textinfo="label+percent",
    showlegend=False
), row=1, col=2)

fig.update_layout(
    title_text="Employee Attrition — Full Dataset Overview",
    template=TEMPLATE, height=430
)
fig.show()


> **Finding:** 47.5% of employees left — roughly **3x the typical industry benchmark of ~15%**.  
> This is not noise. It signals a structural retention problem that demands investigation.


### 3.2 · Age — Are Younger Employees Leaving More?

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Age Distribution by Status", "Median Age by Status"]
)

for status in ["Stayed", "Left"]:
    sub = df[df["attrition"] == status]
    fig.add_trace(go.Histogram(
        x=sub["age"], name=status,
        marker_color=COLORS[status], opacity=0.72, nbinsx=22
    ), row=1, col=1)
    fig.add_trace(go.Box(
        x=sub["attrition"], y=sub["age"],
        name=status, marker_color=COLORS[status], showlegend=False
    ), row=1, col=2)

fig.update_layout(
    barmode="overlay", template=TEMPLATE,
    title_text="Age Distribution vs Attrition",
    legend_title="Status", height=430
)
fig.show()


> **Finding:** Employees who left skew slightly younger, with the highest churn volume in the **25–40 bracket**.  
> Early-career employees are the most mobile and have the most outside options.


### 3.3 · Monthly Income — Does Pay Drive Attrition?

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Income Distribution (Violin)", "Income Spread (Box)"]
)

for status in ["Stayed", "Left"]:
    sub = df[df["attrition"] == status]
    fig.add_trace(go.Violin(
        y=sub["monthly_income"], name=status,
        box_visible=True, meanline_visible=True,
        fillcolor=COLORS[status], line_color=COLORS[status], opacity=0.7
    ), row=1, col=1)
    fig.add_trace(go.Box(
        y=sub["monthly_income"], name=status,
        marker_color=COLORS[status], showlegend=False
    ), row=1, col=2)

fig.update_layout(
    template=TEMPLATE,
    title_text="Monthly Income vs Attrition",
    height=430
)
fig.show()


> **Finding:** Employees who left consistently earned less than those who stayed.  
> Lower income groups exhibit higher turnover — **compensation is a retention lever, not just a recruitment one**.


### 3.4 · Overtime — The Burnout Signal

In [ ]:
grp = group_attrition("overtime")

fig = px.bar(
    grp, x="overtime", y="rate",
    color="overtime",
    color_discrete_map={"Yes": COLORS["Left"], "No": COLORS["Stayed"]},
    text=grp["rate"].apply(lambda x: f"{x}%"),
    labels={"rate": "Attrition Rate (%)", "overtime": "Works Overtime"},
    title="Attrition Rate · Overtime vs Non-Overtime Workers",
    template=TEMPLATE, height=420
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False, yaxis_range=[0, 100])
fig.show()


> **Finding:** Overtime workers show a dramatically higher attrition rate.  
> This is the sharpest binary split in the entire dataset —  
> **overtime is the clearest burnout and exit signal HR has available**.


### 3.5 · Job Satisfaction — The Disengagement-to-Exit Pipeline

In [ ]:
grp = group_attrition("job_satisfaction")

fig = px.bar(
    grp, x="job_satisfaction", y="rate",
    color="rate",
    color_continuous_scale="RdYlGn_r",
    text=grp["rate"].apply(lambda x: f"{x}%"),
    labels={"rate": "Attrition Rate (%)", "job_satisfaction": "Job Satisfaction"},
    title="Attrition Rate by Job Satisfaction Level",
    template=TEMPLATE, height=440,
    category_orders={"job_satisfaction": ORDINALS["job_satisfaction"]}
)
fig.update_traces(textposition="outside")
fig.update_layout(yaxis_range=[0, 100])
fig.show()


> **Finding:** The gradient is unmistakable — attrition falls as satisfaction rises.  
> Employees with **Low satisfaction leave at nearly double the rate** of those with Very High satisfaction.  
> Job satisfaction is both a warning sign and a prevention lever.


### 3.6 · Work-Life Balance

In [ ]:
grp = group_attrition("work_life_balance")

fig = px.bar(
    grp, x="work_life_balance", y="rate",
    color="rate",
    color_continuous_scale="RdYlGn_r",
    text=grp["rate"].apply(lambda x: f"{x}%"),
    labels={"rate": "Attrition Rate (%)", "work_life_balance": "Work-Life Balance"},
    title="Attrition Rate by Work-Life Balance",
    template=TEMPLATE, height=440,
    category_orders={"work_life_balance": ORDINALS["work_life_balance"]}
)
fig.update_traces(textposition="outside")
fig.update_layout(yaxis_range=[0, 100])
fig.show()


> **Finding:** Poor work-life balance is a strong attrition predictor.  
> Together with overtime (3.4), this makes clear: **workload and schedule control are central to retention**.


### 3.7 · Job Role — Which Department Has the Biggest Retention Problem?

In [ ]:
grp = group_attrition("job_role").sort_values("rate", ascending=True)

fig = px.bar(
    grp, y="job_role", x="rate",
    orientation="h",
    color="rate",
    color_continuous_scale="RdYlBu_r",
    text=grp["rate"].apply(lambda x: f"{x}%"),
    labels={"rate": "Attrition Rate (%)", "job_role": "Job Role"},
    title="Attrition Rate by Department / Job Role",
    template=TEMPLATE, height=440
)
fig.update_traces(textposition="outside")
fig.update_layout(xaxis_range=[0, 100])
fig.show()


> **Finding:** Attrition rates vary meaningfully across departments.  
> Roles with more external market demand show elevated churn —  
> **one-size-fits-all retention won't work; each department needs its own strategy**.


### 3.8 · Remote Work — Does Flexibility Improve Retention?

In [ ]:
grp = group_attrition("remote_work")

fig = px.bar(
    grp, x="remote_work", y="rate",
    color="remote_work",
    color_discrete_map={"Yes": COLORS["Stayed"], "No": COLORS["Left"]},
    text=grp["rate"].apply(lambda x: f"{x}%"),
    labels={"rate": "Attrition Rate (%)", "remote_work": "Remote Work"},
    title="Attrition Rate · Remote vs On-Site Employees",
    template=TEMPLATE, height=420
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False, yaxis_range=[0, 100])
fig.show()


> **Finding:** Remote workers show lower attrition than on-site employees.  
> Workplace flexibility is a meaningful retention lever —  
> especially for roles where remote work is operationally feasible.


### 3.9 · Promotions — Does Career Stagnation Drive Exits?

In [ ]:
grp = group_attrition("number_of_promotions").sort_values("number_of_promotions")

fig = px.line(
    grp, x="number_of_promotions", y="rate",
    markers=True,
    labels={"rate": "Attrition Rate (%)", "number_of_promotions": "Promotions Received"},
    title="Attrition Rate vs Number of Promotions",
    template=TEMPLATE, height=430
)
fig.update_traces(
    line_color=COLORS["Left"],
    marker_size=9,
    marker_color=COLORS["Left"]
)
fig.update_layout(yaxis_range=[0, 100])
fig.show()


> **Finding:** Employees with **zero promotions have the highest attrition rate**.  
> The trend falls as promotions increase — career progression is one of the strongest retention signals here.  
> Employees who feel stuck, leave.


### 3.10 · Distance from Home — The Commute Tax

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Commute Distance Distribution", "Distance Spread by Status"]
)

for status in ["Stayed", "Left"]:
    sub = df[df["attrition"] == status]
    fig.add_trace(go.Histogram(
        x=sub["distance_from_home"], name=status,
        marker_color=COLORS[status], opacity=0.72, nbinsx=25
    ), row=1, col=1)
    fig.add_trace(go.Box(
        x=sub["attrition"], y=sub["distance_from_home"],
        name=status, marker_color=COLORS[status], showlegend=False
    ), row=1, col=2)

fig.update_layout(
    barmode="overlay", template=TEMPLATE,
    title_text="Distance from Home vs Attrition",
    legend_title="Status", height=430
)
fig.show()


> **Finding:** Employees who left tended to live farther from the workplace.  
> Long commutes are daily friction that compounds into burnout and eventually exit.  
> Remote flexibility is especially valuable for high-distance employees.


### 3.11 · Job Level — Does Seniority Predict Stability?

In [ ]:
grp = group_attrition("job_level")

fig = px.bar(
    grp, x="job_level", y="rate",
    color="rate",
    color_continuous_scale="RdYlGn_r",
    text=grp["rate"].apply(lambda x: f"{x}%"),
    labels={"rate": "Attrition Rate (%)", "job_level": "Job Level"},
    title="Attrition Rate by Job Level",
    template=TEMPLATE, height=430,
    category_orders={"job_level": ORDINALS["job_level"]}
)
fig.update_traces(textposition="outside")
fig.update_layout(yaxis_range=[0, 100])
fig.show()


> **Finding:** Entry-level employees show the highest attrition — lower pay, less career clarity, fewer switching costs.  
> Senior employees are the most stable.  
> **The first 12–18 months are the highest-risk window.**


### 3.12 · Years at Company — When Do People Leave?

In [ ]:
fig = px.box(
    df, x="attrition", y="years_at_company",
    color="attrition",
    color_discrete_map=COLORS,
    labels={"years_at_company": "Years at Company", "attrition": "Status"},
    title="Years at Company · Leavers vs Stayers",
    template=TEMPLATE, height=430
)
fig.update_layout(showlegend=False)
fig.show()


> **Finding:** Employees who left have fewer years at the company on average.  
> Early-tenure employees (< 3 years) are the highest-risk group.  
> Structured onboarding and first-year engagement programs could significantly reduce early churn.


### 3.13 · Numeric Correlation with Attrition
Which numeric features have the strongest linear association with leaving?


In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ("employee_id", "attrition_flag")]

corr = (
    df[numeric_cols + ["attrition_flag"]]
    .corr()["attrition_flag"]
    .drop("attrition_flag")
    .sort_values()
)

bar_colors = [COLORS["Left" if v > 0 else "Stayed"] for v in corr.values]

fig = go.Figure(go.Bar(
    y=corr.index,
    x=corr.values,
    orientation="h",
    marker_color=bar_colors,
    text=[f"{v:.3f}" for v in corr.values],
    textposition="outside"
))
fig.update_layout(
    title="Pearson Correlation with Attrition Flag",
    xaxis_title="Correlation Coefficient",
    yaxis_title="Feature",
    template=TEMPLATE, height=460
)
fig.show()


> **Finding:**
> - Most associated with **leaving** (positive): distance from home, number of dependents  
> - Most associated with **staying** (negative): monthly income, years at company, number of promotions  
>
> This serves as a ranked shortlist of levers HR can pull for retention programs.


## 4 · Key Findings & HR Recommendations

| # | Finding | Recommended Action |
|---|---------|-------------------|
| 1 | **Overtime** is the sharpest single attrition predictor | Cap mandatory overtime; introduce compensatory time-off |
| 2 | **Low job satisfaction** nearly doubles attrition risk | Regular 1-on-1s; satisfaction pulse surveys; clear growth paths |
| 3 | **Entry-level employees** churn at the highest rate | Structured onboarding + mentorship; 6-month and 1-year milestones |
| 4 | **Remote workers stay longer** | Expand hybrid/remote eligibility where operationally feasible |
| 5 | **Zero promotions = high attrition** | Review promotion cadence; create visible, achievable career ladders |
| 6 | **Lower income correlates with leaving** | Benchmark salaries; prioritize raises in high-attrition roles |
| 7 | **Long commutes add up** | Remote flexibility for high-distance employees |

---
*Dataset: Synthetic Employee Attrition · 74,498 records · Patterns are realistic but not real-world data.*  
*Stack: Pandas · NumPy · Plotly · Python*  
*Kayfa AI & Data Analytics Internship — 2025*
